# Performance Comparison with Lispminer

Python version

In [ ]:
import sys
sys.version_info

System Information

* Processor:	AMD Ryzen 7 PRO 4750U with Radeon Graphics, 1700 Mhz, 8 Core(s), 16 Logical Processor(s)
* OS Name:	Microsoft Windows 11 Enterprise
* System Model:	Lenovo 20UD0013CK
* Installed Physical Memory (RAM)	16.0 GB

In [ ]:
import platform
platform.processor()

## LispMiner Setting:
Different values for support (BASE) and confidence (PIM) were tried.

![lispminerresults](./lisp.png)

## Package action-rules Setting
LispMiner generates the action rules in both ways for the target (Yes -> No, No -> Yes). To get the same behaviour in action-rules package, the mining is run twice. And post filtering is needed because of flexible attributes in LISp-Miner are strictly flexible.

In [ ]:
import pandas as pd

stable_attributes = ["gender", "SeniorCitizen", "Partner"]
flexible_attributes = ["PhoneService",
                       "InternetService",
                       "OnlineSecurity",
                       "DeviceProtection",
                       "TechSupport",
                       "StreamingTV"]
target = 'Churn'
min_stable_attributes = 3
min_flexible_attributes = 2 
undesired_state = 'Yes'
desired_state = 'No'

pd.set_option('display.max_columns', None)
data_frame = pd.read_csv("./../data/telco.csv", sep=";")

In [ ]:
from action_rules import ActionRules

def mining(support, confidence):
    rules = []
    # first run Yes -> No
    action_rules = ActionRules(
        min_stable_attributes=min_stable_attributes,
        min_flexible_attributes=min_flexible_attributes,
        min_undesired_support=support,
        min_undesired_confidence=confidence,
        min_desired_support=support,
        min_desired_confidence=confidence,
        verbose=False
    )
    action_rules.fit(
        data=data_frame,
        stable_attributes=stable_attributes,
        flexible_attributes=flexible_attributes,
        target=target,
        target_undesired_state=undesired_state,
        target_desired_state=desired_state,
        use_gpu=False,
    )
    rules = action_rules.get_rules().get_ar_notation()
    # second run No -> Yes
    action_rules = ActionRules(
        min_stable_attributes=min_stable_attributes,
        min_flexible_attributes=min_flexible_attributes,
        min_undesired_support=support,
        min_undesired_confidence=confidence,
        min_desired_support=support,
        min_desired_confidence=confidence,
        verbose=False
    )
    action_rules.fit(
        data=data_frame,
        stable_attributes=stable_attributes,
        flexible_attributes=flexible_attributes,
        target=target,
        target_undesired_state=desired_state,
        target_desired_state=undesired_state,
        use_gpu=False,
    )
    rules += action_rules.get_rules().get_ar_notation()
    rul = []
    for r in rules:
        if '*' not in r: # just strictly flexible
            rul.append(r)
    return rul

In [ ]:
from actionrules.actionRulesDiscovery import ActionRulesDiscovery

def miningARAS(support, confidence):
    rules = []
    actionRulesDiscovery = ActionRulesDiscovery()
    actionRulesDiscovery.load_pandas(data_frame)
    actionRulesDiscovery.fit(
        stable_attributes=stable_attributes,
        flexible_attributes=flexible_attributes,
        consequent=target,
        conf=confidence*100,
        supp=-support,
        desired_changes=[[undesired_state, desired_state], [desired_state, undesired_state]],
        min_stable_attributes=min_stable_attributes,
        min_flexible_attributes=min_flexible_attributes,
        is_reduction=True
    )
    rules = actionRulesDiscovery.get_action_rules_representation()
    rul = []
    for r in rules:
        if '*' not in r: # just strictly flexible
            rul.append(r)
    return rul

# Results

### Support 70, confidence 50%

LISp-Miner

![lispminerresults](./performance/70_50_1.png)
![lispminerresults](./performance/70_50_2.png)
![lispminerresults](./performance/70_50_3.png)
![lispminerresults](./performance/70_50_4.png)
![lispminerresults](./performance/70_50_5.png)
![lispminerresults](./performance/70_50_6.png)
![lispminerresults](./performance/70_50_7.png)

In [ ]:
import numpy as np
time_70_50 = [35, 27, 27, 29, 27, 27, 28]
print('Average')
average = np.mean(time_70_50)
print(average)
print('Std. dev.')
std_dev = np.std(time_70_50)
print(std_dev)

action-rules

In [ ]:
%timeit mining(70, 0.5)

In [ ]:
len(mining(70, 0.5))

actionrules-lukassy (ARAS)

In [ ]:
%timeit miningARAS(70, 0.5)

In [ ]:
len(miningARAS(70, 0.5))

### Support 70, confidence 60%

LISp-Miner

![lispminerresults](./performance/70_60_1.png)
![lispminerresults](./performance/70_60_2.png)
![lispminerresults](./performance/70_60_3.png)
![lispminerresults](./performance/70_60_4.png)
![lispminerresults](./performance/70_60_5.png)
![lispminerresults](./performance/70_60_6.png)
![lispminerresults](./performance/70_60_7.png)

In [ ]:
time_70_60 = [28, 24, 25, 24, 24, 34, 24]
print('Average')
average = np.mean(time_70_60)
print(average)
print('Std. dev.')
std_dev = np.std(time_70_60)
print(std_dev)

action-rules

In [ ]:
%timeit mining(70, 0.6)

In [ ]:
len(mining(70, 0.6))

actionrules-lukassy (ARAS)

In [ ]:
%timeit miningARAS(70, 0.6)

In [ ]:
len(miningARAS(70, 0.6))

### Support 140, confidence 50%

LISp-Miner

![lispminerresults](./performance/140_50_1.png)
![lispminerresults](./performance/140_50_2.png)
![lispminerresults](./performance/140_50_3.png)
![lispminerresults](./performance/140_50_4.png)
![lispminerresults](./performance/140_50_5.png)
![lispminerresults](./performance/140_50_6.png)
![lispminerresults](./performance/140_50_7.png)

In [ ]:
time_140_50 = [12, 12, 12, 11, 11, 11, 12]
print('Average')
average = np.mean(time_140_50)
print(average)
print('Std. dev.')
std_dev = np.std(time_140_50)
print(std_dev)

action-rules

In [ ]:
%timeit mining(140, 0.5)

In [ ]:
len(mining(140, 0.5))

actionrules-lukassy (ARAS)

In [ ]:
%timeit miningARAS(140, 0.5)

In [ ]:
len(miningARAS(140, 0.5))

### Support 140, confidence 60%

LISp-Miner

![lispminerresults](./performance/140_60_1.png)
![lispminerresults](./performance/140_60_2.png)
![lispminerresults](./performance/140_60_3.png)
![lispminerresults](./performance/140_60_4.png)
![lispminerresults](./performance/140_60_5.png)
![lispminerresults](./performance/140_60_6.png)
![lispminerresults](./performance/140_60_7.png)

In [ ]:
time_140_60 = [9, 8, 10, 10, 11, 11, 11]
print('Average')
average = np.mean(time_140_60)
print(average)
print('Std. dev.')
std_dev = np.std(time_140_60)
print(std_dev)

action-rules

In [ ]:
%timeit mining(140, 0.6)

In [ ]:
len(mining(140, 0.6))

actionrules-lukassy (ARAS)

In [ ]:
%timeit miningARAS(140, 0.6)

In [ ]:
len(miningARAS(140, 0.6))

## Compare rules

LISp-Miner

action-rules

In [ ]:
mining(140, 0.6)

# Results Table

In [ ]:
import pandas as pd
data = {
    'Support, Confidence': ['70, 50%', '70, 60%', '140, 50%', '140, 60%'],
    'LISp-Miner Time (s)': [28.571, 26.143, 11.571, 10.0],
    'LISp-Miner Rules': [178, 32, 70, 8],
    'Action-Rules Time (s)': [0.996, 1.070, 0.480, 0.491],
    'Action-Rules Rules': [178, 32, 70, 8],
    'Actionrules-Lukassykora Time (s)': [65, 13, 29.9, 5.52],
    'Actionrules-Lukassykora Rules': [178, 32, 70, 8],
}

# Create DataFrame
df = pd.DataFrame(data)

#df['Action-Rules Speed (x)'] = (df['LISp-Miner Time (s)']) / df['Action-Rules Time (s)']
df